# spaCy vs NLTK — Setup and Comparison
This notebook installs spaCy and NLTK, then demonstrates sentence and word tokenization side-by-side to highlight key behavioral differences between the two libraries.

### Install spaCy
spaCy must be installed before importing. The `pip install spacy` magic command handles this inside Jupyter. On first run, a kernel restart may be required.

In [1]:
pip install spacy

### Download the spaCy English Model
`en_core_web_sm` is a lightweight pre-trained English model (~12 MB). It bundles a tokenizer, tagger, parser, NER, and lemmatizer. The `!` prefix runs this as a shell command inside Jupyter — omitting it causes a `SyntaxError`.

In [6]:
!python -m spacy download en_core_web_sm


### Install NLTK
NLTK (Natural Language Toolkit) is a general-purpose NLP toolkit focused on research and education. Its modules are imported separately — unlike spaCy's unified pipeline, NLTK is a collection of independent tools.

In [ ]:
pip install nltk

## Part 1: Sentence and Word Tokenization in spaCy
spaCy uses a statistical dependency parser to set sentence boundaries. It correctly recognizes `Dr.` as an honorific abbreviation rather than a sentence-ending period — because the model has learned this from real English text corpora.

Load the pre-trained model and process a text string. `nlp(text)` runs the full pipeline and returns a `Doc` object — spaCy's central data container holding annotated tokens, sentences, entities, and more.

In [8]:
import spacy

In [10]:
nlp = spacy.load("en_core_web_sm")
doc = nlp("Dr. Strange loves pav bhaji of mumbai. Hulk loves chat of delhi")

`doc.sents` is a generator over sentence spans. spaCy correctly detects 2 sentences, treating `Dr.` as an abbreviation. Compare this with NLTK below — which incorrectly splits on it.

In [12]:
for sentence in doc.sents:
    print(sentence)

Nested iteration gives a two-level view: sentence -> word. Punctuation (`.`) is a separate token — this is intentional behavior enabling downstream POS and NER tasks that need punctuation signals.

In [14]:
for sentence in doc.sents:
    for word in sentence:
        print(word)

## Part 2: Sentence and Word Tokenization in NLTK
NLTK's `sent_tokenize` uses the Punkt algorithm — a rule-based, unsupervised model trained to detect sentence boundaries. Download `punkt_tab` (required once) before using it.

`nltk.download('punkt_tab')` fetches the pre-trained Punkt tokenizer data and caches it in `~/nltk_data`. This is only needed once per machine. The function returns `True` on success.

In [21]:
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt_tab')

Key difference: NLTK's Punkt treats `Dr.` as a sentence boundary, producing 3 segments instead of 2. This is a known limitation — Punkt uses period-based heuristics and struggles with abbreviations not in its training set.

In [23]:
sent_tokenize("Dr. Strange loves pav bhaji of mumbai. Hulk loves chat of delhi")

`word_tokenize` uses the Penn Treebank tokenization rules. It separates contractions (`don't` -> `do`, `n't`) and handles punctuation differently from spaCy.

In [25]:
from nltk.tokenize import word_tokenize

NLTK splits `Dr.` into `['Dr', '.']` — separating the abbreviation from its period. spaCy keeps `Dr.` as a single token. Neither is strictly wrong; the difference reflects their design philosophy (rule-based vs statistical).

In [27]:
word_tokenize("Dr. Strange loves pav bhaji of mumbai. Hulk loves chat of delhi")

## Summary: NLTK vs spaCy at a Glance

| Dimension | NLTK | spaCy |
|---|---|---|
| **Philosophy** | Research and teaching toolkit | Production-ready NLP |
| **Ease of use** | Manual pipeline assembly | Pre-built, opinionated pipeline |
| **Speed** | Slower (pure Python) | Faster (Cython-compiled) |
| **Pre-trained models** | No built-in DL models | Yes — NER, POS, parsing |
| **Tokenization** | Rule-based (Punkt / Treebank) | Statistical + rule-based |
| **NER** | Requires Stanford NER or manual build | Built-in, state-of-the-art |
| **Best for** | Learning and experimentation | Real-world NLP applications |

Use NLTK to understand NLP concepts; use spaCy to build NLP systems.